In [48]:
import pandas as pd 
import os

In [49]:
base_path = "data/IEMOCAP_full_release/"

data_path = "dialog/EmoEvaluation/"
wav_path = "sentences/wav"

In [50]:
import re

list_features = ["Start_Time", "End_Time", "Turn_Name", "Emotion", "Valence", "Arousal", "Dominance"]
df = pd.DataFrame([], columns=list_features)
for i in range(1,6):
    new_path = os.path.join(base_path,f'Session{i}/',data_path)
    for entry in os.listdir(new_path):
        if entry.endswith(".txt"):

            # Read the text file
            with open(new_path+entry, "r") as file:
                lines = file.readlines()

            # Regular expression to match lines starting with [START_TIME - END_TIME]
            pattern = r"\[(\d+\.\d+) - (\d+\.\d+)\]\s+(\S+)\s+(\S+)\s+\[(\d+\.\d+),\s*(\d+\.\d+),\s*(\d+\.\d+)\]"

            # List to hold the extracted data
            data = []

            # Loop through each line and match with the pattern
            for line in lines:
                match = re.match(pattern, line)
                if match:
                    start_time = float(match.group(1))
                    end_time = float(match.group(2))
                    turn_name = match.group(3)
                    emotion = match.group(4)
                    valence = float(match.group(5))
                    arousal = float(match.group(6))
                    dominance = float(match.group(7))
                    
                    # Append extracted data to the list
                    data.append([start_time, end_time, turn_name, emotion, valence, arousal, dominance])
            # Create a DataFrame from the extracted data
            temp_df = pd.DataFrame(data, columns=list_features)
            df = pd.concat([df, temp_df], ignore_index=True)

# Display the DataFrame
print(df)


       Start_Time  End_Time            Turn_Name Emotion  Valence  Arousal  \
0            6.77    8.4600  Ses01M_impro01_F000     ang      1.5      3.5   
1            8.55   10.6000  Ses01M_impro01_F001     ang      2.5      3.5   
2           14.46   18.2300  Ses01M_impro01_F002     xxx      3.0      3.0   
3           19.48   23.4300  Ses01M_impro01_F003     xxx      3.0      3.0   
4           23.43   26.4675  Ses01M_impro01_F004     fru      2.5      3.5   
...           ...       ...                  ...     ...      ...      ...   
10034      252.71  255.5000  Ses05M_impro04_M037     fru      2.5      3.0   
10035      262.01  264.0000  Ses05M_impro04_M038     fru      3.0      3.0   
10036      264.79  266.3100  Ses05M_impro04_M039     fru      2.5      3.0   
10037      267.83  270.4200  Ses05M_impro04_M040     ang      3.0      3.0   
10038      273.80  275.8900  Ses05M_impro04_M041     sad      2.0      2.5   

       Dominance  
0            4.5  
1            3.5  
2     

In [51]:
import os
import librosa

# Initialize an empty DataFrame
wav_df = pd.DataFrame([], columns=["Turn_Name", "wav_file"])

# Iterate through the sessions
for i in range(1, 6):
    session_path = os.path.join(base_path, f"Session{i}/", wav_path)
    tmp_path_list = os.listdir(session_path)
    
    # Iterate through directories within the session path
    for dir in tmp_path_list:
        dir_path = os.path.join(session_path, dir)
        
        # Check if it's a directory before proceeding
        if os.path.isdir(dir_path):
            for file in os.listdir(dir_path):
                
                # Only proceed if the file is a .wav file
                if file.endswith(".wav"):
                    file_path = os.path.join(dir_path, file)
                    
                    # Load the .wav file using librosa
                    audio, sample_rate = librosa.load(file_path, sr=16000)
                    # Add the audio array to the DataFrame
                    wav_df = pd.concat([wav_df, pd.DataFrame([{"Turn_Name": file.replace(".wav",""), "wav_file" :audio}])])

# Display the DataFrame
print(wav_df)


                 Turn_Name                                           wav_file
0      Ses01M_impro01_F022  [-0.021148682, -0.027923584, -0.041534424, -0....
0      Ses01M_impro01_M003  [0.012420654, 0.009552002, 0.0073242188, 0.003...
0      Ses01M_impro01_M017  [0.0045166016, 0.0052490234, 0.0063476562, 0.0...
0      Ses01M_impro01_M016  [0.0027160645, 0.0014953613, -0.0025939941, -0...
0      Ses01M_impro01_M002  [0.0074157715, 0.0008239746, 0.00024414062, 0....
..                     ...                                                ...
0   Ses05F_script02_1_F002  [0.011871338, 0.015716553, 0.019134521, 0.0207...
0   Ses05F_script02_1_M023  [0.0010375977, 0.0015563965, 0.0022583008, 0.0...
0   Ses05F_script02_1_M022  [-0.0050964355, -0.0024719238, -0.00015258789,...
0   Ses05F_script02_1_F003  [-0.001739502, -0.001739502, -0.0017089844, -0...
0   Ses05F_script02_1_F017  [-0.00045776367, -0.00064086914, -0.0002441406...

[10039 rows x 2 columns]


In [52]:
new_df = df.join(wav_df.set_index("Turn_Name"), "Turn_Name").drop(columns=["Start_Time", "End_Time", "Emotion"])
new_df

,Turn_Name,Valence,Arousal,Dominance,wav_file
0,Ses01M_impro01_F000,1.5,3.5,4.5,"[0.005432129, 0.0048828125, 0.0053710938, 0.00..."
1,Ses01M_impro01_F001,2.5,3.5,3.5,"[0.0014343262, 0.00030517578, -0.0016784668, -..."
2,Ses01M_impro01_F002,3.0,3.0,3.5,"[0.028442383, 0.02355957, 0.017974854, 0.01376..."
3,Ses01M_impro01_F003,3.0,3.0,3.0,"[0.002319336, 0.0021362305, -0.0008239746, -0...."
4,Ses01M_impro01_F004,2.5,3.5,3.5,"[0.0019836426, 0.0051879883, 0.0010070801, -0...."
...,...,...,...,...,...
10034,Ses05M_impro04_M037,2.5,3.0,3.0,"[0.003326416, 0.00088500977, 0.0010986328, 0.0..."
10035,Ses05M_impro04_M038,3.0,3.0,3.5,"[-0.0036621094, -0.0038146973, -0.0039367676, ..."
10036,Ses05M_impro04_M039,2.5,3.0,3.5,"[-0.0019226074, -0.0011291504, -0.00076293945,..."
10037,Ses05M_impro04_M040,3.0,3.0,3.5,"[-0.00039672852, -0.0008239746, -0.001159668, ..."


In [53]:
def normalize(col, min, max):
    return (col - min)/(max - min)

In [54]:
new_df["Arousal"] = normalize(new_df["Arousal"], 0, 5)

new_df["Valence"] = normalize(new_df["Valence"], 0, 5)

new_df["Dominance"] = normalize(new_df["Dominance"], 0, 5)

new_df

,Turn_Name,Valence,Arousal,Dominance,wav_file
0,Ses01M_impro01_F000,0.3,0.7,0.9,"[0.005432129, 0.0048828125, 0.0053710938, 0.00..."
1,Ses01M_impro01_F001,0.5,0.7,0.7,"[0.0014343262, 0.00030517578, -0.0016784668, -..."
2,Ses01M_impro01_F002,0.6,0.6,0.7,"[0.028442383, 0.02355957, 0.017974854, 0.01376..."
3,Ses01M_impro01_F003,0.6,0.6,0.6,"[0.002319336, 0.0021362305, -0.0008239746, -0...."
4,Ses01M_impro01_F004,0.5,0.7,0.7,"[0.0019836426, 0.0051879883, 0.0010070801, -0...."
...,...,...,...,...,...
10034,Ses05M_impro04_M037,0.5,0.6,0.6,"[0.003326416, 0.00088500977, 0.0010986328, 0.0..."
10035,Ses05M_impro04_M038,0.6,0.6,0.7,"[-0.0036621094, -0.0038146973, -0.0039367676, ..."
10036,Ses05M_impro04_M039,0.5,0.6,0.7,"[-0.0019226074, -0.0011291504, -0.00076293945,..."
10037,Ses05M_impro04_M040,0.6,0.6,0.7,"[-0.00039672852, -0.0008239746, -0.001159668, ..."


In [55]:
import IPython.display as ipd
import random

rand_int = random.randint(0, len(new_df))
print(new_df.iloc[rand_int])

ipd.Audio(data=new_df["wav_file"][rand_int], autoplay=True, rate=16000)

Turn_Name                               Ses02M_script01_1_M015
Valence                                                    0.5
Arousal                                                    0.6
Dominance                                                  0.7
wav_file     [0.00018310547, -0.0012817383, -0.0020141602, ...
Name: 2933, dtype: object
